# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadFaizan0023/FlyRank_ML_internship_repo/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
data = pd.read_csv("data_for_baseline_action_score.csv")

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*



**Prioritization Logic**<br>
Our queue prioritizes content pages experiencing the most severe performance decay by sorting them descending by `decline_score`. This ensures reviewers can target high-risk assets first with directional, evidence-backed recommendations.

**Why a Human Can Trust This Queue**<br>
This ranking is constructed directly from actual, observed performance changes rather than black-box assumptions. We compare the following five core metrics from `prev30` (previous 30 days) to `last30` (last 30 days) to build empirical trust:

1. **Click-Through Rate (`ctr_prev30` vs. `ctr_last30`)**: A drop in CTR indicates that although our search impressions might remain steady, our titles and snippets are failing to capture searcher clicks.
2. **Engagement Rate (`engagement_rate_prev30` vs. `engagement_rate_last30`)**: A contraction in user interaction signals that visitors who click through find the content less relevant or satisfying than they used to.
3. **Cost Per Click (`cpc_prev30` vs. `cpc_last30`)**: Changes in CPC indicate shift in monetization potential, commercial intent, or ad-market competitiveness of the keyword themes.
4. **Average Position (`gsc_avg_position_prev30d` vs. `gsc_avg_position_last30d`)**: Slipping downwards in ranking signals that competitor content is fresher or better aligned with user search queries.
5. **Traffic & Impressions Drop**: Contrasting variables like `gsc_impressions_prev30d` vs. `gsc_impressions_last30d` helps confirm whether the entire search volume of a topic has shrunk, or if we are simply losing our market share.

In [6]:
ranked_queue = data.sort_values('decline_score', ascending=False)

In [4]:
ranked_queue[:20]

,client_hash_id,content_hash_id,gsc_impressions_prev30d,gsc_clicks_prev30d,gsc_avg_position_prev30d,ga4_pageviews_prev30d,ga4_sessions_prev30d,ga4_users_prev30d,'ga4_engaged_session_prev30,ga4_total_engagement_sec_prev30,...,sessions_paid_last30,sessions_ai_last30,scroll_events_last30,cpc_last30,backlinks_last30,ctr_last30,engagement_rate_last30,scroll_events_rate_last30,days_since_update_last30,decline_score
141355,client_20259bd6705d81d4,content_bff6079940ef8faf,513,7,20.725146,8,7,6,0,0,...,0,0,0,0.00,0.0,0.004292,0.0,0.000000,47,17
141356,client_20259bd6705d81d4,content_dbf35ea262d38fb3,1254,5,23.271132,7,7,7,0,0,...,0,0,0,0.00,0.0,0.002564,0.0,0.000000,47,17
141357,client_20259bd6705d81d4,content_ba1b42d3d9cf77d0,90,1,4.388889,3,3,3,0,1,...,0,0,1,0.00,0.0,0.000000,0.0,1.000000,47,17
141358,client_20259bd6705d81d4,content_bff6079940ef8faf,513,7,20.725146,8,7,6,0,0,...,0,0,2,0.00,0.0,0.010893,0.0,0.333333,47,17
141359,client_20259bd6705d81d4,content_bff6079940ef8faf,513,7,20.725146,8,7,6,0,0,...,0,0,0,0.00,0.0,0.002304,0.0,0.000000,47,17
141328,client_20259bd6705d81d4,content_030054976ef226d1,471,5,10.350318,4,4,4,0,0,...,0,0,0,0.00,0.0,0.009132,0.0,0.000000,47,17
141329,client_23a62021009f63c4,content_7ff47b1364f9351d,929,6,16.503767,7,7,7,3,290,...,0,0,0,0.00,0.0,0.000000,0.0,0.000000,18,17
141330,client_23a62021009f63c4,content_7ff47b1364f9351d,929,6,16.503767,7,7,7,3,290,...,0,0,0,0.00,0.0,0.000000,0.0,0.000000,18,17
141331,client_23a62021009f63c4,content_6d159b4985b766e8,42,1,12.142857,2,1,1,0,0,...,0,0,0,0.00,0.0,0.000000,0.0,0.000000,11,17
141332,client_20259bd6705d81d4,content_40827fb92be7f5bf,379,9,5.620053,8,7,7,0,0,...,0,0,1,0.00,0.0,0.010870,0.0,0.500000,47,17


In [5]:
ranked_queue.columns

Index(['client_hash_id', 'content_hash_id', 'gsc_impressions_prev30d',
       'gsc_clicks_prev30d', 'gsc_avg_position_prev30d',
       'ga4_pageviews_prev30d', 'ga4_sessions_prev30d', 'ga4_users_prev30d',
       ''ga4_engaged_session_prev30', 'ga4_total_engagement_sec_prev30',
       'sessions_organic_prev30', 'sessions_direct_prev30',
       'sessions_referral_prev30', 'sessions_social_prev30',
       'sessions_paid_prev30', 'sessions_ai_prev30', 'scroll_events_prev30',
       'content_created_date', 'content_updated_date', 'cpc_prev30',
       'backlinks_prev30', 'ctr_prev30', 'engagement_rate_prev30',
       'scroll_events_rate_prev30', 'days_since_update_prev30',
       'gsc_impressions_last30d', 'gsc_clicks_last30d',
       'gsc_avg_position_last30d', 'ga4_pageviews_last30d',
       'ga4_sessions_last30d', 'ga4_users_last30d',
       ''ga4_engaged_session_last30', 'ga4_total_engagement_sec_last30',
       'sessions_organic_last30', 'sessions_direct_last30',
       'sessions_referr

In [8]:
# Calculate and display summary statistics for the top decline targets to verify signals
top_queue = ranked_queue.head(20)

# Let's inspect the key metrics of the highest decline-score pages
summary_stats = top_queue[[
    'decline_score',
    'ga4_pageviews_prev30d',
    'ga4_pageviews_last30d',
    'gsc_impressions_prev30d',
    'gsc_impressions_last30d',
    'gsc_clicks_prev30d',
    'gsc_clicks_last30d',
    'gsc_avg_position_prev30d',
    'gsc_avg_position_last30d',
    'ga4_sessions_prev30d',
    'ga4_sessions_last30d',
    'ga4_users_prev30d',
    'ga4_users_last30d',
    "'ga4_engaged_session_prev30",
    "'ga4_engaged_session_last30",
    'ga4_total_engagement_sec_prev30',
    'ga4_total_engagement_sec_last30',
    'sessions_organic_prev30',
    'sessions_organic_last30',
    'sessions_direct_prev30',
    'sessions_direct_last30',
    'sessions_referral_prev30',
    'sessions_referral_last30',
    'sessions_social_prev30',
    'sessions_social_last30',
    'sessions_paid_prev30',
    'sessions_paid_last30',
    'sessions_ai_prev30',
    'sessions_ai_last30',
    'scroll_events_prev30',
    'scroll_events_last30',
    'cpc_prev30',
    'cpc_last30',
    'backlinks_prev30',
    'backlinks_last30',
    'ctr_prev30',
    'ctr_last30',
    'engagement_rate_prev30',
    'engagement_rate_last30',
    'scroll_events_rate_prev30',
    'scroll_events_rate_last30',
    'days_since_update_last30'
]].describe()

display(summary_stats)

,decline_score,ga4_pageviews_prev30d,ga4_pageviews_last30d,gsc_impressions_prev30d,gsc_impressions_last30d,gsc_clicks_prev30d,gsc_clicks_last30d,gsc_avg_position_prev30d,gsc_avg_position_last30d,ga4_sessions_prev30d,...,cpc_last30,backlinks_prev30,backlinks_last30,ctr_prev30,ctr_last30,engagement_rate_prev30,engagement_rate_last30,scroll_events_rate_prev30,scroll_events_rate_last30,days_since_update_last30
count,20.0,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,...,20.000000,16.00,16.00,20.000000,20.000000,20.000000,20.0,20.000000,20.000000,20.000000
mean,17.0,4.700000,1.950000,792.450000,332.950000,4.000000,0.900000,10.935738,17.376915,4.250000,...,0.200000,29.75,29.75,0.011741,0.003796,0.064286,0.0,0.266071,0.091667,35.100000
std,0.0,2.451637,1.316894,657.776356,253.842155,2.554665,1.209611,7.030755,8.678358,2.468219,...,0.768895,119.00,119.00,0.013815,0.004762,0.157006,0.0,0.295071,0.250584,14.552627
min,17.0,2.000000,1.000000,42.000000,21.000000,1.000000,0.000000,3.038674,5.224490,1.000000,...,0.000000,0.00,0.00,0.000519,0.000000,0.000000,0.0,0.000000,0.000000,11.000000
25%,17.0,2.750000,1.000000,430.750000,133.250000,1.000000,0.000000,5.189656,9.860265,2.750000,...,0.000000,0.00,0.00,0.005841,0.000000,0.000000,0.0,0.000000,0.000000,23.250000
50%,17.0,4.000000,2.000000,513.000000,327.000000,3.500000,1.000000,8.138329,15.004536,3.500000,...,0.000000,0.00,0.00,0.006696,0.002329,0.000000,0.0,0.250000,0.000000,47.000000
75%,17.0,7.000000,2.000000,1010.250000,427.250000,6.000000,1.000000,16.503767,22.778852,7.000000,...,0.000000,0.00,0.00,0.013645,0.006074,0.000000,0.0,0.446429,0.000000,47.000000
max,17.0,8.000000,6.000000,1925.000000,1035.000000,9.000000,5.000000,23.271132,35.360731,7.000000,...,3.420000,476.00,476.00,0.062500,0.016129,0.428571,0.0,1.000000,1.000000,49.000000


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

The people in SEO can use this as a decision-support for speedy working.

This proxy label is a prioritization signal, not an autonomous decision system. It ranks content by likelihood of decline based on decline_score, a proxy label built from 17 rule-based conditions (search visibility, CTR, engagement, traffic across channels, and others) comparing a prev30 to a last30 window. Its intended use is to help a human reviewer decide which pages to look at first, and not to trigger automatic content changes, and not to diagnose the specific cause of decline.
Moreover, this these signals are measured from the difference in signals of 2 months, so it has limitation to very long duration content pages.

In [9]:
# Calculate and display the score distribution to prove the queue operates as a prioritization signal
score_counts = ranked_queue['decline_score'].value_counts().sort_index(ascending=False)
score_percentages = (ranked_queue['decline_score'].value_counts(normalize=True) * 100).sort_index(ascending=False)

# Create a summary table to display the density of priorities
priority_distribution = pd.DataFrame({
    'Count': score_counts,
    'Percentage (%)': score_percentages
})

print("Decline Score Distribution (Prioritization Density):")
display(priority_distribution.head(10))

# Show the threshold boundary behavior (proving it acts as a smooth ranking tool rather than binary decision-making)

print(f"\nTotal pages evaluated: {len(ranked_queue):,}")
print(f"Pages with severe decay (Decline Score >= 15): {len(ranked_queue[ranked_queue['decline_score'] >= 15]):,}")

Decline Score Distribution (Prioritization Density):


,Count,Percentage (%)
decline_score,,
17,212004,5.082353
16,649399,15.567984
15,720274,17.267064
14,643934,15.436972
13,573049,13.737652
12,450887,10.809074
11,349326,8.374361
10,234951,5.632459
9,135773,3.254874



Total pages evaluated: 4,171,375
Pages with severe decay (Decline Score >= 15): 1,581,677


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

The decision to refresh page and actually refresh page automatically should never be automated. Because this model just predicts the score from possible signals that only helps in making the decision if page needs a refresh or not.
A person has to check the specific SEO or other content page problems himself and resolve it himself.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.